### imports & functions

In [1]:
import pandas as pd
import numpy as np
import MDAnalysis as mda
import matplotlib.pyplot as plt
from itertools import combinations
import sys
from MDAnalysis.analysis import contacts
from itertools import product
import itertools
import mdtraj as md
from contact_map import ContactFrequency

In [2]:
path_git='../../cascade_computing' #todo - path to git
sys.path.append(path_git)
from src.compute.utils import toolbox_interactions as interactiontools

/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/Bio/Application/__init__.py:39: BiopythonDeprecationWarning: The Bio.Application modules and modules relying on it have been deprecated.

Due to the on going maintenance burden of keeping command line application
wrappers up to date, we have decided to deprecate and eventually remove these
modules.

We instead now recommend building your command line and invoking it directly
with the subprocess module.
  warnings.warn(


In [3]:
def empty_contact_df():
    """Create an empty DataFrame with the correct schema for contacts."""
    meta_dict = {
        'frame': pd.Series([], dtype='int64'),
        'n_cont': pd.Series([], dtype='int64'),
        'cont_a': pd.Series([], dtype='object'),
        'cont_b': pd.Series([], dtype='object'),
        'q_values': pd.Series([], dtype='object'),
    }

    if bool_bonds:
        meta_dict.update({
            'cation_pi': pd.Series([], dtype='object'),
            'pi_stacking': pd.Series([], dtype='object'),
            'hbond': pd.Series([], dtype='object'),
            'salt_bridge': pd.Series([], dtype='object')
        })

    return pd.DataFrame(meta_dict)

In [4]:
bool_debug=False
bool_bonds=True
bool_q=False
def process_frame_chunk_for_split(top, traj, group_a_sel, group_b_sel, d_max, radius, k, chunk_size):
    """
    Process a chunk of frames to calculate contacts and specific interactions.
    
    Parameters
    ----------
    top, traj : str
    group_a_sel, group_b_sel : str
    d_max, radius : float
    k, chunk_size : int
    
    Returns
    -------
    pandas.DataFrame
    """

    #match = re.search(r'chunk_(\d+)', traj)
    #if match:
    #    # Convert 1-based file index to 0-based calculation index
    #    k = int(match.group(1)) - 1
    #else:
    #    print('check traj part', traj, flush=True)
        
    if bool_debug:
        worker_id = get_worker()
        print('part index',k, traj, flush=True)
        print('WORKER_ID', worker_id.name, flush=True)
        print(f"{worker_id.memory_manager.memory_limit / 1024**3:.2f} GB", flush=True)
        print(f"[{worker_id}] memory before processing {traj}: {psutil.Process().memory_info().rss / 1024**2:.2f} MB", flush=True)

    u = mda.Universe(top, traj)#, format='xtc', topology_format='pdb')

    str_a = group_a_sel + " and not name H*"
    u_group_a_sel = u.select_atoms(group_a_sel) 
    group_a = u_group_a_sel.select_atoms("not name H*")

    str_b = group_b_sel + " and not name H*"
    group_b = u.select_atoms(str_b) 
    
    if bool_bonds:
        group_a_sc_H = """
            ((resname SER and (name HG HG1)) or
            (resname THR and name HG1) or
            (resname TYR and name HH) or
            (resname ASN and (name HD21 HD22)) or
            (resname GLN and (name HE21 HE22)) or
            (resname HIS HSD HSE HSP and (name HD1 HE2)) or
            (resname TRP and name HE1) or
            (resname LYS and (name HZ1 HZ2 HZ3)) or
            (resname ARG and (name HE HH11 HH12 HH21 HH22)))
        """

        #for h-bonds - expand selection to hydrogens 
        #str_c=group_a_sel+ " and " +group_a_sc_H
        #group_c = u.select_atoms(str_c)
        #group_c=u_group_a_sel.select_atoms(group_a_sc_H)
        #print("c_selection", set(group_c.atoms.names), len(group_c.atoms.ids), flush=True)

        #selecting specific atoms participating in bonds
        cation_pi_selection = interactiontools.get_interaction_sel('cation_pi', group_a, group_b)
        pi_stacking_selection = interactiontools.get_interaction_sel('pi_stacking', group_a, group_b)
        #hbond_selection=interactiontools.get_interaction_sel('hbond', group_a,group_b)
        salt_bridge_selection = interactiontools.get_interaction_sel('salt_bridge', group_a, group_b)

    data = {
        'frame': [], 'n_cont': [], 'cont_a': [], 'cont_b': [], 'q_values': [] 
    }

    if bool_bonds:
        data['cation_pi'] = []
        data['pi_stacking'] = []
        data['hbond'] = []
        data['salt_bridge'] = []

    for ts in u.trajectory:
        frm = ts.frame
        dims = ts.dimensions
        frm_abs = k * chunk_size + frm

        cm_A = group_a.center_of_mass(wrap=True, unwrap=False, compound='group')
        cm_B = group_b.center_of_mass(wrap=True, unwrap=False, compound='group')
        dist_AB = contacts.distance_array(cm_A[None, :], cm_B[None, :], box=dims)[0, 0]
        
        if dist_AB < d_max * dims[2]:
            dist = contacts.distance_array(group_a, group_b, box=dims)
            mat_contacts = contacts.contact_matrix(dist, radius)
            n_contacts = mat_contacts.sum()

            if n_contacts > 0:
                pairs = np.array(np.where(mat_contacts))

                cont_a_ids_u = group_a.atoms.ids[pairs[0, :]]
                cont_b_ids_u = group_b.atoms.ids[pairs[1, :]]
                cont_a_rids_u = group_a.atoms.resids[pairs[0, :]]
                cont_b_rids_u = group_b.atoms.resids[pairs[1, :]]

                data['frame'].append(frm_abs)
                data['n_cont'].append(n_contacts)
                data['cont_a'].append(cont_a_ids_u.tolist())
                data['cont_b'].append(cont_b_ids_u.tolist())
                
                # Native contacts (q-values) calculation
                q_values_dict = {}
                if bool_q: 
                    pattern = os.path.join(output_path, "*_reference.pkl")
                    matching_files = glob.glob(pattern)
    
                    for file in matching_files:
                        ref_val = pd.read_pickle(file)
                        try:                        
                            ind_nat_A = np.asarray(ref_val["nc_a_ind"])
                            ind_nat_B = np.asarray(ref_val["nc_b_ind"])
                            r0 = np.asarray(ref_val["nc_dist"]).flatten()
                            prots = ref_val["prots"]
                            #print("selections", len(group_a), len(group_b), flush=True)
        
                            if len(ind_nat_A) == 0 or len(ind_nat_B) == 0:
                                print(f"Skipping {prots}: empty reference contact set")
                                continue
                    
                            if np.max(ind_nat_A) >= dist.shape[0] or np.max(ind_nat_B) >= dist.shape[1]:
                                #print(f"Skipping {prots}: indices out of bounds for current group")
                                continue
                                
                            r = dist[ind_nat_A, ind_nat_B]
                            q = np.mean(1.0 / (1 + np.exp(BETA_CONST * (r - LAMBDA_CONST * r0))))
                            q_values_dict[prots] = float(q)
                        except:
                            print('no ' + ref_val["prots"] + ' interface')
                data['q_values'].append(q_values_dict)

                # Interaction profiles 
                if bool_bonds:
                    l1 = list(set(cont_a_rids_u))
                    l2 = list(set(cont_b_rids_u))
                    all_cont_resids = l1 + l2

                    all_chains = (group_a + group_b)
                    all_resids = all_chains.residues.resids

                    def eval_candidate_pairs(candiate_pairs, all_cont_resids, all_chains, all_resids):
                        """
                        Filter candidate atom pairs to ensure they belong to residues in contact.
                        
                        Parameters
                        ----------
                        candiate_pairs : array-like
                        all_cont_resids : list
                        all_chains : AtomGroup
                        all_resids : array-like
                        
                        Returns
                        -------
                        numpy.ndarray
                        """
                        mask_subset = np.isin(np.array(candiate_pairs), all_cont_resids).all(axis=1)
                        filtered_pairs = np.array(candiate_pairs)[mask_subset]

                        indices_a = np.searchsorted(all_resids, np.array(filtered_pairs)[:,0])
                        indices_b = np.searchsorted(all_resids, np.array(filtered_pairs)[:,1])
                        mols_a = all_chains.residues[indices_a]
                        mols_b = all_chains.residues[indices_b]
                        mol_pairs = np.column_stack((mols_a, mols_b))

                        return mol_pairs

                    #print(cation_pi_selection)
                    cation_pi_candidates = eval_candidate_pairs(cation_pi_selection, all_cont_resids, all_chains, all_resids)
                    cation_pi_contacts = interactiontools.cation_pi_contact(cation_pi_candidates, distance_cutoff=6.0, angle_cutoff=60.0)

                    #print(pi_stacking_selection)
                    pi_stacking_candidates = eval_candidate_pairs(pi_stacking_selection, all_cont_resids, all_chains, all_resids)
                    pi_stacking_contacts = interactiontools.pi_stacking_contact(pi_stacking_candidates, distance_cutoff=7.0, angle_cutoff=30.0, psi_cutoff=45.0)

                    #print(salt_bridge_selection)
                    salt_bridge_candidates = eval_candidate_pairs(salt_bridge_selection, all_cont_resids, all_chains, all_resids)
                    salt_bridge_contacts = interactiontools.salt_bridge_contact(salt_bridge_candidates, distance_cutoff=4.0)

               
                    #hbond #todo
                    #for donor -h
                    #mask_30 = np.isin(l_sel3[0].ids, cont_a_ids_u)

                    #for hydrogens -h

                    #atoms in contact (cont_a_ids_u) but from atom selection including hydrogens (u_group_a_sel)
                    #mask_30_c = np.isin(u_group_a_sel.ids, cont_a_ids_u)

                    #all atoms of the corresponding residues
                    #group_a_c=u_group_a_sel[mask_30_c].residues.atoms

                    #but only take the hydrogens from group_a_c
                    #group_c=group_a_c.select_atoms(group_a_sc_H)

                    #for aceptors -
                    #mask_31 = np.isin(l_sel3[1].ids, cont_b_ids_u)

                    
                    #hbond for later
                    #bool_hbond =interactiontools.hbond_contact(l_sel3[0][mask_30], group_c, l_sel3[1][mask_31], distance_cutoff=3.5, angle_cutoff=30.0)


                    data['cation_pi'].append(cation_pi_contacts)
                    data['pi_stacking'].append(pi_stacking_contacts)
                    data['hbond'].append([]) 
                    data['salt_bridge'].append(salt_bridge_contacts)
                    
    if bool_debug:
        print(f"[{worker_id}] RSS before trim: {psutil.Process().memory_info().rss / 1024 ** 2:.2f} MB", flush=True)

    #save memory
    #del u, group_a, group_b, ts
    #release_memory()
    
    if bool_debug:
        print(f"{worker_id.memory_manager.memory_limit / 1024**3:.2f} GB", flush=True)
        print(f"[{worker_id}] memory after processing {traj}: {psutil.Process().memory_info().rss / 1024**2:.2f} MB", flush=True)

    
    if not data['frame']:
        return empty_contact_df()

    return pd.DataFrame(data)  

### load data 

In [6]:
#original data
top=path_git+'/examples/data/replica_6/postprocessing/test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4_pi_clean_10_chains.pdb'
xtc=path_git+'/examples/data/replica_6/postprocessing/test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4_full_pi_ref_debug_10_chains_2001.xtc'
sys_domains=pd.read_parquet(path_git+'/examples/data/replica_6/postprocessing/test_project_MUT16_65_46af39bf2445d01c22c25aadf9d305c4_sys_domains.parquet')

In [7]:
u = mda.Universe(top)
traj=top

/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "


In [8]:
u.residues.resids

array([   1,    2,    3, ..., 3610, 3611, 3612], shape=(1892,))

### 10 chain pairs - test for one frame

In [9]:
#pick chains
ALL_MOLS_SEL = [f"resid {row['min']}-{row['max']}" for _, row in sys_domains.iterrows()]
combinations_in = list(itertools.combinations(enumerate(ALL_MOLS_SEL), 2))[10:20]
combinations_in

[((0, 'resid 1-172'), (11, 'resid 1893-2064')),
 ((0, 'resid 1-172'), (12, 'resid 2065-2236')),
 ((0, 'resid 1-172'), (13, 'resid 2237-2408')),
 ((0, 'resid 1-172'), (14, 'resid 2409-2580')),
 ((0, 'resid 1-172'), (15, 'resid 2581-2752')),
 ((0, 'resid 1-172'), (16, 'resid 2753-2924')),
 ((0, 'resid 1-172'), (17, 'resid 2925-3096')),
 ((0, 'resid 1-172'), (18, 'resid 3097-3268')),
 ((0, 'resid 1-172'), (19, 'resid 3269-3440')),
 ((0, 'resid 1-172'), (20, 'resid 3441-3612'))]

In [10]:
#contact parameters
d_max=10
radius=6

#we take the initial chunk
k=0
chunk_size=0

In [11]:
#results
l_df=[]

In [12]:
for a, b in combinations_in:
    print(k)
    group_a_sel=a[1]
    group_b_sel=b[1]
    l_df.append(process_frame_chunk_for_split(top, traj, group_a_sel, group_b_sel, d_max, radius, k, chunk_size))
    k=k+1

0
1
2


/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "


3


/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "


4
5


...

/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "


7


/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "


8
9


/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "


In [13]:
full_df=pd.concat(l_df, ignore_index=True)
full_df

,frame,n_cont,cont_a,cont_b,q_values,cation_pi,pi_stacking,hbond,salt_bridge
0,0,1451,"[624, 626, 626, 628, 628, 628, 628, 628, 628, ...","[4555, 4555, 4557, 4551, 4554, 4555, 4557, 455...",{},"[[103, 2055], [104, 2055]]",[],[],"[[104, 2053], [1995, 97], [1984, 98]]"
1,0,113,"[84, 84, 85, 85, 85, 88, 88, 88, 88, 88, 91, 9...","[10892, 10894, 10890, 10892, 10894, 10888, 108...",{},[],[],[],[]
2,0,674,"[35, 35, 35, 37, 37, 37, 37, 37, 37, 37, 40, 4...","[23062, 23173, 23185, 23062, 23171, 23173, 231...",{},[],"[[18, 3233]]",[],[]


In [14]:
full_df.columns

Index(['frame', 'n_cont', 'cont_a', 'cont_b', 'q_values', 'cation_pi',
       'pi_stacking', 'hbond', 'salt_bridge'],
      dtype='object')

In [15]:
#load data frame
#full_df[['frame', 'n_cont', 'cont_a', 'cont_b', 'cation_pi',
#       'pi_stacking', 'hbond', 'salt_bridge']].to_parquet('./combined_data2.parquet', index=False)

In [16]:
#full_df=pd.read_parquet('./combined_data.parquet')
#full_df

#### pi_stacking

In [17]:
pi_stacking = [
    list(outer_item) 
    for outer_item in full_df['pi_stacking'].values 
    if len(outer_item) > 0
]

In [18]:
pi_stacking

[[[18, 3233]]]

In [19]:
l_pi_stacking=[]
for item in pi_stacking:
    for i in item:
        l_pi_stacking.append(list(i))

l_pi_stacking

[[18, 3233]]

#### cation_pi

In [20]:
cation_pi = [
    list(outer_item) 
    for outer_item in full_df['cation_pi'].values 
    if len(outer_item) > 0
]
cation_pi

[[[103, 2055], [104, 2055]]]

In [21]:
l_cation_pi=[]
for item in cation_pi:
    for i in item:
        l_cation_pi.append(list(i))

In [22]:
l_cation_pi

[[103, 2055], [104, 2055]]

#### salt bridge

In [23]:
salt_bridge = [
    list(outer_item) 
    for outer_item in full_df['salt_bridge'].values 
    if len(outer_item) > 0
]

In [24]:
l_salt_bridge=[]
for item in salt_bridge:
    for i in item:
        l_salt_bridge.append(list(i))

l_salt_bridge

[[104, 2053], [1995, 97], [1984, 98]]

### comparison contact map explorer - 10 chains

In [25]:
traj = md.load(top, top=top)
topology = traj.topology
topology

<mdtraj.Topology with 1 chains, 1892 residues, 28776 atoms, 29446 bonds at 0x7fc633042f80>

In [26]:
# PDB sequence numbers (case-sensitive: resSeq)
resseqs = [res.resSeq for res in traj.top.residues]

# Internal 0-based indices
resids = [res.index for res in traj.top.residues]

# Residue names (e.g., 'ALA', 'GLY')
resnames = [res.name for res in traj.top.residues]

In [27]:
combinations_in

[((0, 'resid 1-172'), (11, 'resid 1893-2064')),
 ((0, 'resid 1-172'), (12, 'resid 2065-2236')),
 ((0, 'resid 1-172'), (13, 'resid 2237-2408')),
 ((0, 'resid 1-172'), (14, 'resid 2409-2580')),
 ((0, 'resid 1-172'), (15, 'resid 2581-2752')),
 ((0, 'resid 1-172'), (16, 'resid 2753-2924')),
 ((0, 'resid 1-172'), (17, 'resid 2925-3096')),
 ((0, 'resid 1-172'), (18, 'resid 3097-3268')),
 ((0, 'resid 1-172'), (19, 'resid 3269-3440')),
 ((0, 'resid 1-172'), (20, 'resid 3441-3612'))]

In [28]:
l_unique_pairs=[]
for a,b in combinations_in:
    #print(a,b)
    a_chain=a[1].replace("resid", "resSeq").replace("-", " to ")
    b_chain=b[1].replace("resid", "resSeq").replace("-", " to ")

    chain_a_indices = traj.top.select(a_chain+" and element != H")
    chain_b_indices = traj.top.select(b_chain+"and element != H")

    #compare with MDAnalysis selection
    u = mda.Universe(top, top)#, format='xtc', topology_format='pdb')
    str_a = a[1] + " and not name H*"
    u_group_a_sel = u.select_atoms(group_a_sel) 
    group_a = u_group_a_sel.select_atoms("not name H*")

    str_b = b[1] + " and not name H*"
    group_b = u.select_atoms(str_b) 

    #compare the two atom selections
    # 1. Convert MDTraj 0-based indices to 1-based IDs
    mdtraj_set_a = set(chain_a_indices + 1)
    mdtraj_set_b = set(chain_b_indices + 1)
    
    # 2. Get MDAnalysis IDs
    mda_set_a = set(group_a.atoms.ids)
    mda_set_b = set(group_b.atoms.ids)
    
    # 3. Compare them
    if mdtraj_set_a != mda_set_a:
        print(f"Mismatch in Group A! MDTraj: {len(mdtraj_set_a)} atoms, MDA: {len(mda_set_a)} atoms")
        print(f"Atoms in MDTraj but NOT MDA: {mdtraj_set_a - mda_set_a}")
        print(f"Atoms in MDA but NOT MDTraj: {mda_set_a - mdtraj_set_a}")
    
    if mdtraj_set_b != mda_set_b:
        print(f"Mismatch in Group B! MDTraj: {len(mdtraj_set_b)} atoms, MDA: {len(mda_set_b)} atoms")


    extra_ids = list(mdtraj_set_a - mda_set_a)

    if extra_ids:
        print(f"First 10 'Extra' Atoms found by MDTraj:")
        for eid in extra_ids[:10]:
            # Remember: eid is 1-based, MDTraj top is 0-based
            atom = traj.top.atom(eid - 1)
            print(f"ID: {eid} | Name: {atom.name} | Res: {atom.residue.name}{atom.residue.resSeq} | Chain: {atom.residue.chain.index}")
    
    #calculate inter-chain contacts
    frame_contacts = ContactFrequency(
        traj, 
        query=chain_a_indices, 
        haystack=chain_b_indices, 
        cutoff=0.6
    )
    
    sparse_contacts = frame_contacts.atom_contacts.sparse_matrix
    
    # Get the atom indices of the non-zero entries
    rows, cols = sparse_contacts.nonzero()
    
    unique_pairs = [(int(i) + 1, int(j) + 1) if i < j else (int(j) + 1, int(i) + 1) for i, j in zip(rows, cols)]
    l_unique_pairs=l_unique_pairs+unique_pairs


/gpfs/fs1/home/lubaltz/p_3_10/lib/python3.10/site-packages/MDAnalysis/topology/PDBParser.py:350: UserWarning: Element information is missing, elements attribute will not be populated. If needed these can be guessed using universe.guess_TopologyAttrs(context='default', to_guess=['elements']).
  warnings.warn("Element information is missing, elements attribute "


In [29]:
l_unique_pairs

[(1302, 3885),
 (1302, 3885),
 (1472, 3974),
 (1472, 3974),
 (1292, 3880),
 (1292, 3880),
 (1360, 4161),
 (1360, 4161),
 (1364, 4167),
 (1364, 4167),
 (1272, 3885),
 (1272, 3885),
 (866, 4484),
 (866, 4484),
 (684, 4537),
 (684, 4537),
 (1472, 4103),
 (1472, 4103),
 (1317, 3914),
 (1317, 3914),
 (1471, 4004),
 (1471, 4004),
 (1468, 4007),
 (1468, 4007),
 (1196, 3890),
 (1196, 3890),
 (1571, 5065),
 (1571, 5065),
 (958, 4253),
 (958, 4253),
 (1692, 3108),
 (1692, 3108),
 (869, 4434),
 (869, 4434),
 (1323, 4124),
 (1323, 4124),
 (959, 4247),
 (959, 4247),
 (686, 4545),
 (686, 4545),
 (671, 4539),
 (671, 4539),
 (1302, 3867),
 (1302, 3867),
 (956, 4253),
 (956, 4253),
 (1353, 4105),
 (1353, 4105),
 (869, 4491),
 (869, 4491),
 (1519, 5099),
 (1519, 5099),
 (817, 4513),
 (817, 4513),
 (1312, 3914),
 (1312, 3914),
 (628, 4559),
 (628, 4559),
 (682, 4537),
 (682, 4537),
 (1519, 5091),
 (1519, 5091),
 (813, 4515),
 (813, 4515),
 (1526, 5099),
 (1526, 5099),
 (1288, 3879),
 (1288, 3879),
 (673,

In [30]:
l_cont_a=[]
for item in full_df['cont_a'].values:
    print(type(item))
    l_cont_a=l_cont_a+item

l_cont_b=[]
for item in full_df['cont_b'].values:
    print(type(item))
    l_cont_b=l_cont_b+item

<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>
<class 'list'>


In [31]:
#atms_a=full_df['cont_a'].values.flatten().tolist()[0]
#atms_b=full_df['cont_b'].values.flatten().tolist()[0]
unique_atoms = [(i, j) if i < j else (j, i) for i, j in zip(l_cont_a, l_cont_b)]


In [32]:
set_a = set(l_unique_pairs)
set_b = set(unique_atoms)

# 1. Contacts present in BOTH (Common)
common = set_a.intersection(set_b)

# 2. Contacts in A but NOT in B (exp has cc not)
lost = set_a - set_b

# 3. Contacts in B but NOT in A (cc has exp not)
gained = set_b - set_a

In [33]:
len(set_a)

2238

In [34]:
len(set_b)

2238

In [35]:
len(common)

2238

In [36]:
len(gained)

0

In [37]:
lost

set()

In [ ]:
### comparison get-contacts - todo

In [ ]:
### comparison hbonds - todo